## Импорты

In [1]:
try:
    import google.colab
    %pip install pandas torch transformers datasets evaluate sacrebleu bert-score sentence-transformers rapidfuzz deep-translator tqdm numpy matplotlib PyYAML pymorphy3
    %pip install --upgrade tiktoken protobuf sentencepiece
    %pip install accelerate>=1.1.0
    is_colab = True
except:
    is_colab = False

if is_colab:
    !git clone https://github.com/AbonentVneSeti/text_processing_final_project
    %cd text_processing_final_project

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 130.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 35.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.12.0
    Uninstalling tiktoken-0.12.0:

Cloning into 'text_processing_final_project'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 90 (delta 42), reused 72 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 6.76 MiB | 10.45 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/text_processing_final_project


In [2]:
import os
import sys
sys.path.append('..')
from utils import load_config
import pandas as pd
import importlib

config = load_config("config.yaml")
print("Конфиг загружен")

Конфиг загружен


## Создание датасета

### Backtranslation

In [ ]:
if not os.path.exists("data/paraphrases_raw.parquet"):
    from dataset.create_dataset import tapaco, backtranslation

    #tapaco
    tapaco_params = config["source_params"]["tapaco"]
    df_tapaco = tapaco.load_or_create(tapaco_params)
    
    unique_originals = df_tapaco[['original']].drop_duplicates()['original'].tolist()
    unique_originals = unique_originals

    #backtranslation
    bt_params = config["source_params"]["backtranslation"].copy()
    bt_params['sentences'] = unique_originals
    df_bt = backtranslation.load_or_create(bt_params)

    #save
    output_path = "data/paraphrases_raw.parquet"
    df_bt.to_parquet(output_path, index=False)
    print(f"Датасет сохранён в {output_path}")

## Предобработка

In [ ]:
#df_bt = pd.read_parquet("data/backtranslated.parquet")
df_bt = pd.read_parquet("data/paraphrases_raw.parquet")
print(f"Загружено {len(df_bt)} пар")

Загружено 187100 пар


In [4]:
from dataset.prepare_dataset.prepare import prepare_dataset

df_clean = prepare_dataset(df_bt, config["preprocessing"])
print(f"После очистки: {len(df_clean)} пар")

Running filter_by_length:  43%|████▎     | 3/7 [00:03<00:05,  1.37s/step]    /usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/828k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Running filter_semantic_similarity: 100%|██████████| 7/7 [06:27<00:00, 55.39s/step] 

После очистки: 101945 пар


## Сохранение

In [5]:
output_path = "data/paraphrases_clean.parquet"
df_clean.to_parquet(output_path, index=False)
print(f"Датасет сохранён в {output_path}")

Датасет сохранён в data/paraphrases_clean.parquet


In [7]:
if is_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp {output_path} /content/drive/MyDrive/
    print("Скопировано на Google Drive")

Mounted at /content/drive
Скопировано на Google Drive
